# Retrieval, Reranking, and Generation

This notebook exposes a single `Retriever` that can be pointed at any chunking strategy (`fixed`, `section`, `semantic`, `hierarchical`) and switched between retrieval modes: `vector`, `bm25`, `hybrid` (Reciprocal Rank Fusion of dense + sparse), and `hybrid+rerank` (cross-encoder on top).

The evaluation notebook drives this module over the full chunking x retrieval matrix; the cells below are for quick interactive sanity checks.

In [1]:
import importlib
import json
import os
import pickle
import re
import time
from dataclasses import dataclass, field
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PROCESSED = ROOT / 'data' / 'processed'
ARTIFACTS = ROOT / 'artifacts'

EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
RERANKER_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'


def clean_query(query: str) -> str:
    return re.sub(r'\s+', ' ', query or '').strip()


_BM25_STOPWORDS = frozenset({
    'a', 'an', 'the', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'be',
    'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will',
    'would', 'could', 'should', 'may', 'might', 'shall', 'can', 'that',
    'this', 'these', 'those', 'it', 'its', 'we', 'our', 'they', 'their',
    'he', 'she', 'his', 'her', 'not', 'no', 'nor', 'so', 'yet', 'both',
    'each', 'such', 'than', 'too', 'very', 'also', 'into', 'through',
    'during', 'including', 'between', 'however', 'thereof', 'therein',
    'herein', 'pursuant', 'hereof', 'thereunder',
})


def _tokenize_bm25_query(query: str) -> list[str]:
    tokens = clean_query(query).lower().split()
    return [token for token in tokens if token not in _BM25_STOPWORDS and len(token) > 1]


def make_openai_llm(model: str = 'gpt-4o-mini'):
    api_key = os.getenv('OPENAI_API_KEY')
    if not api_key:
        raise RuntimeError('OPENAI_API_KEY is not set. Add it to your environment before making LLM calls.')

    openai_module = importlib.import_module('openai')
    client = openai_module.OpenAI(api_key=api_key)

    def llm_generate(prompt: str) -> str:
        response = client.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0,
        )
        return (response.choices[0].message.content or '').strip()

    return llm_generate


def _build_line_offsets(path: Path) -> list[int]:
    offsets: list[int] = []
    with path.open('rb') as handle:
        while True:
            offset = handle.tell()
            line = handle.readline()
            if not line:
                break
            offsets.append(offset)
    return offsets


@dataclass
class StrategyStore:
    strategy: str
    metadata: pd.DataFrame
    jsonl_path: Path
    line_offsets: list[int]
    faiss_index: faiss.Index
    bm25: BM25Okapi
    row_cache: dict[int, dict] = field(default_factory=dict, repr=False)

    @classmethod
    def load(cls, strategy: str) -> 'StrategyStore':
        metadata = pd.read_json(ARTIFACTS / f'{strategy}_chunk_metadata.json')
        metadata = metadata.reset_index(drop=True)
        jsonl_path = DATA_PROCESSED / f'chunks_{strategy}.jsonl'
        line_offsets = _build_line_offsets(jsonl_path)
        faiss_index = faiss.read_index(str(ARTIFACTS / f'{strategy}_faiss.index'))
        with (ARTIFACTS / f'{strategy}_bm25.pkl').open('rb') as handle:
            bm25 = pickle.load(handle)
        return cls(
            strategy=strategy,
            metadata=metadata,
            jsonl_path=jsonl_path,
            line_offsets=line_offsets,
            faiss_index=faiss_index,
            bm25=bm25,
        )

    def _hydrate_row(self, row_index: int) -> dict:
        if row_index in self.row_cache:
            return self.row_cache[row_index]
        if row_index < 0 or row_index >= len(self.line_offsets):
            raise IndexError(f'Chunk index out of range for strategy {self.strategy}: {row_index}')

        with self.jsonl_path.open('rb') as handle:
            handle.seek(self.line_offsets[row_index])
            payload = json.loads(handle.readline())

        payload.update(self.metadata.iloc[row_index].to_dict())
        self.row_cache[row_index] = payload
        return payload

    def hydrate(self, indices: np.ndarray | list[int]) -> pd.DataFrame:
        records = []
        for index in np.asarray(indices).ravel().tolist():
            if index < 0 or index >= len(self.line_offsets):
                continue
            records.append(self._hydrate_row(int(index)).copy())
        if not records:
            return self.metadata.iloc[0:0].copy()
        rows = pd.DataFrame(records)
        metadata_columns = [column for column in self.metadata.columns if column in rows.columns]
        extra_columns = [column for column in rows.columns if column not in metadata_columns]
        return rows[metadata_columns + extra_columns].reset_index(drop=True)


@dataclass
class Retriever:
    embedding_model: SentenceTransformer
    reranker: CrossEncoder | None = None
    stores: dict[str, StrategyStore] = field(default_factory=dict)

    def store(self, strategy: str) -> StrategyStore:
        if strategy not in self.stores:
            self.stores[strategy] = StrategyStore.load(strategy)
        return self.stores[strategy]

    def vector_search(self, query: str, strategy: str, k: int) -> pd.DataFrame:
        store = self.store(strategy)
        q = self.embedding_model.encode([clean_query(query)], normalize_embeddings=True)
        scores, indices = store.faiss_index.search(np.asarray(q, dtype='float32'), k)
        rows = store.hydrate(indices[0])
        rows['score'] = scores[0]
        rows['retrieval'] = 'vector'
        return rows.reset_index(drop=True)

    def bm25_search(self, query: str, strategy: str, k: int) -> pd.DataFrame:
        store = self.store(strategy)
        tokenized = _tokenize_bm25_query(query)
        scores = store.bm25.get_scores(tokenized)
        top = np.argsort(scores)[::-1][:k]
        rows = store.hydrate(top)
        rows['score'] = np.asarray(scores)[top]
        rows['retrieval'] = 'bm25'
        return rows.reset_index(drop=True)

    def hybrid_search(self, query: str, strategy: str, k: int, pool: int = 20, rrf_c: int = 60) -> pd.DataFrame:
        vec = self.vector_search(query, strategy, k=pool)
        bm = self.bm25_search(query, strategy, k=pool)
        rrf: dict[str, float] = {}
        lookup: dict[str, pd.Series] = {}
        for rank, row in enumerate(vec.itertuples(index=False)):
            rrf[row.chunk_id] = rrf.get(row.chunk_id, 0.0) + 1.0 / (rrf_c + rank + 1)
            lookup[row.chunk_id] = vec.iloc[rank]
        for rank, row in enumerate(bm.itertuples(index=False)):
            rrf[row.chunk_id] = rrf.get(row.chunk_id, 0.0) + 1.0 / (rrf_c + rank + 1)
            lookup.setdefault(row.chunk_id, bm.iloc[rank])
        order = sorted(rrf.items(), key=lambda kv: kv[1], reverse=True)[:k]
        rows = pd.DataFrame([lookup[cid] for cid, _ in order]).reset_index(drop=True)
        rows['score'] = [score for _, score in order]
        rows['retrieval'] = 'hybrid'
        return rows

    def rerank(self, query: str, candidates: pd.DataFrame, top_k: int) -> pd.DataFrame:
        if self.reranker is None or candidates.empty:
            return candidates.head(top_k).reset_index(drop=True)
        pairs = [(clean_query(query), t) for t in candidates['text'].tolist()]
        scores = self.reranker.predict(pairs)
        out = candidates.copy()
        out['rerank_score'] = scores
        out = out.sort_values('rerank_score', ascending=False).head(top_k).reset_index(drop=True)
        out['retrieval'] = 'hybrid+rerank'
        return out

    def retrieve(self, query: str, strategy: str, mode: str, k: int = 5, rrf_c: int = 60) -> pd.DataFrame:
        if mode == 'vector':
            return self.vector_search(query, strategy, k=k)
        if mode == 'bm25':
            return self.bm25_search(query, strategy, k=k)
        if mode == 'hybrid':
            pool = max(k * 6, 60)
            return self.hybrid_search(query, strategy, k=k, pool=pool, rrf_c=rrf_c)
        if mode == 'hybrid+rerank':
            pool_inner = max(k * 8, 80)
            candidates = self.hybrid_search(query, strategy, k=pool_inner, pool=pool_inner, rrf_c=rrf_c)
            return self.rerank(query, candidates, top_k=k)
        raise ValueError(f'Unknown retrieval mode: {mode}')

    def timed_retrieve(self, query: str, strategy: str, mode: str, k: int = 5, rrf_c: int = 60) -> tuple[pd.DataFrame, float]:
        start = time.perf_counter()
        hits = self.retrieve(query, strategy, mode, k=k, rrf_c=rrf_c)
        return hits, time.perf_counter() - start


def make_retriever() -> Retriever:
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    try:
        reranker = CrossEncoder(RERANKER_NAME)
    except Exception:
        reranker = None
    return Retriever(embedding_model=embedding_model, reranker=reranker)

In [2]:
def rewrite_query(query: str, llm_generate=None) -> str:
    if llm_generate is None:
        return clean_query(query)
    prompt = (
        'Rewrite the user question for SEC filing retrieval. Keep company names and facts intact.\n'
        f'Question: {query}\nRewritten query:'
    )
    return clean_query(llm_generate(prompt))


def hyde_query(query: str, llm_generate=None) -> str:
    if llm_generate is None:
        return clean_query(query)
    prompt = (
        'Write a short hypothetical answer to this SEC/financial question.\n'
        f'Question: {query}\nHypothetical answer:'
    )
    return clean_query(llm_generate(prompt))


def multi_query_variants(query: str, llm_generate=None) -> list[str]:
    if llm_generate is None:
        return [clean_query(query)]
    prompt = (
        'Create 3 alternative search queries for SEC filing retrieval. Keep them concise.\n'
        f'Question: {query}\nQueries:'
    )
    response = llm_generate(prompt)
    variants = [clean_query(line) for line in response.splitlines() if clean_query(line)]
    return variants[:3] or [clean_query(query)]

In [3]:
def build_context(
    chunks: pd.DataFrame,
    max_tokens: int = 3000,
    use_parent: bool = True,
) -> str:
    """
    Build a context string from retrieved chunks for the LLM prompt.

    If use_parent=True and a 'parent_text' column is present, the full parent
    section is used as context instead of the small child chunk. This gives the
    LLM richer surrounding text while still benefiting from precise child-level
    retrieval.

    Args:
        chunks:     DataFrame of retrieved chunks (output of retriever.retrieve).
        max_tokens: Approximate token ceiling for total context. Default 3000.
                    Tune between 1500-6000 depending on your LLM context window.
        use_parent: If True, use parent_text when available. Default True.

    Returns:
        A formatted context string.
    """
    import tiktoken

    encoder = tiktoken.get_encoding('cl100k_base')

    snippets, total_tokens = [], 0
    seen_parents = set()

    for row in chunks.itertuples(index=False):
        has_parent = use_parent and hasattr(row, 'parent_text') and pd.notna(row.parent_text)
        body = str(row.parent_text).strip() if has_parent else str(row.text).strip()

        dedup_key = body[:120]
        if dedup_key in seen_parents:
            continue
        seen_parents.add(dedup_key)

        snippet = (
            f'[Source: {row.company} | {row.form} | {row.filing_date} | {row.chunk_id}]\n'
            f'{body}'
        )
        snippet_tokens = len(encoder.encode(snippet))
        if total_tokens + snippet_tokens > max_tokens:
            break
        snippets.append(snippet)
        total_tokens += snippet_tokens

    return '\n\n---\n\n'.join(snippets)


def baseline_prompt(query: str, context: str) -> str:
    return (
        'Answer the question using only the provided SEC filing context. If the answer is not in the context, say you do not know.\n\n'
        f'Question: {query}\n\nContext:\n{context}\n\nAnswer:'
    )


def enhanced_prompt(query: str, context: str) -> str:
    return (
        'You are a strict SEC filing analyst. Answer using ONLY the context provided below.\n'
        'Rules:\n'
        '1. Cite the source filing (company, form type, date, and Item number if visible).\n'
        '2. Use specific numbers and facts from the context - do not paraphrase vaguely.\n'
        '3. If the answer spans multiple filings, say so explicitly.\n'
        '4. If the context does not contain a sufficient answer, say: '
        '"The provided context does not contain enough information to answer this question."\n\n'
        f'Question: {query}\n\nContext:\n{context}\n\n'
        'Answer (cite source filings inline):'
    )


def self_refine_answer(query: str, answer: str, context: str, llm_generate=None) -> str:
    if llm_generate is None:
        return answer
    critique = llm_generate(
        'Check this answer for grounding errors, missing facts, or unsupported claims.\n'
        f'Question: {query}\nAnswer: {answer}\nContext: {context}\nCritique:'
    )
    return llm_generate(
        'Revise the answer using the critique below, but do not add facts not in context.\n'
        f'Question: {query}\nAnswer: {answer}\nCritique: {critique}\nContext: {context}\nRevised answer:'
    )

In [4]:
def compare_pipelines(query: str, retriever: Retriever, baseline=('fixed', 'vector'), enhanced=('hierarchical', 'hybrid+rerank'), llm_generate=None, top_k: int = 5) -> dict:
    b_strategy, b_mode = baseline
    e_strategy, e_mode = enhanced
    baseline_hits = retriever.retrieve(query, strategy=b_strategy, mode=b_mode, k=top_k)
    rewritten = rewrite_query(query, llm_generate=llm_generate)
    hyde = hyde_query(rewritten, llm_generate=llm_generate)
    enhanced_hits = retriever.retrieve(hyde, strategy=e_strategy, mode=e_mode, k=top_k)
    baseline_ctx = build_context(baseline_hits)
    enhanced_ctx = build_context(enhanced_hits)
    baseline_output = llm_generate(baseline_prompt(query, baseline_ctx)) if llm_generate else baseline_prompt(query, baseline_ctx)
    enhanced_output = llm_generate(enhanced_prompt(query, enhanced_ctx)) if llm_generate else enhanced_prompt(query, enhanced_ctx)
    return {
        'query': query,
        'baseline': {'config': baseline, 'hits': baseline_hits, 'context': baseline_ctx, 'output': baseline_output},
        'enhanced': {'config': enhanced, 'hits': enhanced_hits, 'context': enhanced_ctx, 'output': enhanced_output},
    }

In [5]:
retriever = make_retriever()

llm_generate = make_openai_llm(model='gpt-4o-mini')


example_query = 'What is the revenue of Visa in 2025 Q3 ?'
demo = compare_pipelines(example_query, retriever=retriever, llm_generate=llm_generate)
demo['baseline']['hits'][['chunk_id', 'company', 'form', 'filing_date']].head()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,chunk_id,company,form,filing_date
0,V-10-K-2025-11-06-fixed-150,Visa Inc.,10-K,2025-11-06
1,V-10-Q-2025-04-30-fixed-134,Visa Inc.,10-Q,2025-04-30
2,V-10-K-2025-11-06-fixed-492,Visa Inc.,10-K,2025-11-06
3,V-10-Q-2025-04-30-fixed-293,Visa Inc.,10-Q,2025-04-30
4,V-10-K-2025-11-06-fixed-590,Visa Inc.,10-K,2025-11-06
